In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_1420_Ashok_Vihar_Delhi_DPCC_1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,182.54,320.04,5.72,24.96,30.68,50.59,6.86,0.97,3.00,...,NaN,10.97,79.42,1.25,58.58,0.00,0.00,96.58,983.07,NaN
1,2024-01-02,172.09,297.51,8.31,26.99,35.29,42.69,8.86,1.08,3.23,...,NaN,10.64,76.40,1.18,58.50,0.00,0.00,131.20,983.13,NaN
2,2024-01-03,174.98,328.21,16.00,39.31,41.43,42.91,10.63,1.66,4.19,...,NaN,10.03,86.98,1.10,58.38,0.00,0.00,73.39,983.16,NaN
3,2024-01-04,202.82,377.96,19.02,45.08,39.44,36.98,16.02,1.28,4.77,...,NaN,10.53,87.64,1.43,58.06,0.00,0.00,37.35,983.00,NaN
4,2024-01-05,157.52,322.79,13.69,41.22,33.05,47.94,13.90,1.35,4.97,...,NaN,11.25,90.44,1.23,57.80,0.00,0.00,21.51,983.00,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,159.48,225.48,6.91,61.36,37.62,84.71,6.04,0.95,14.67,...,NaN,14.69,87.66,1.13,60.19,0.34,0.30,10.64,980.54,NaN
362,2024-12-28,98.71,132.00,12.16,52.54,37.36,32.71,7.78,1.09,6.56,...,NaN,15.04,89.56,1.27,57.05,0.02,0.02,16.74,982.00,NaN
363,2024-12-29,96.54,142.21,1.67,28.78,16.05,33.60,8.95,0.84,20.75,...,NaN,14.65,87.26,1.42,59.36,0.00,0.00,53.47,982.00,NaN
364,2024-12-30,98.04,147.96,2.12,30.44,17.15,30.09,9.02,0.96,27.08,...,NaN,13.29,84.23,1.39,58.95,0.00,0.00,48.88,982.00,NaN


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 21)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Xylene (µg/m³)']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 Timestamp          0
PM2.5 (µg/m³)      0
PM10 (µg/m³)       0
NO (µg/m³)         0
NO2 (µg/m³)        0
NOx (ppb)          0
NH3 (µg/m³)        0
SO2 (µg/m³)        0
CO (mg/m³)         0
Ozone (µg/m³)      0
Benzene (µg/m³)    0
Toluene (µg/m³)    0
AT (°C)            0
RH (%)             0
WS (m/s)           0
WD (deg)           0
RF (mm)            0
TOT-RF (mm)        0
SR (W/mt2)         0
BP (mmHg)          0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (366, 20)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         182.54        320.04        5.72        24.96   
1  2024-01-02         172.09        297.51        8.31        26.99   
2  2024-01-03         174.98        328.21       16.00        39.31   
3  2024-01-04         202.82        377.96       19.02        45.08   
4  2024-01-05         157.52        322.79       13.69        41.22   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  \
0      30.68        50.59         6.86        0.97           3.00   
1      35.29        42.69         8.86        1.08           3.23   
2      41.43        42.91        10.63        1.66           4.19   
3      39.44        36.98        16.02        1.28           4.77   
4      33.05        47.94        13.90        1.35           4.97   

   Benzene (µg/m³)  Toluene (µg/m³)  AT (°C)  RH (%)  WS (m/s)  WD (deg)  \
0             0.59             1.64    10.97   79.42      1

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³),Toluene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg)
0,2024-01-01,1.512373,0.913760,-0.444022,-1.085945,-0.083078,0.696234,-1.270951,-0.300997,-1.177500,-0.816123,-1.647026,-1.760893,0.978405,-0.153981,0.402046,0.0,0.0,-0.309241,0.984706
1,2024-01-02,1.343764,0.720725,-0.115881,-0.983717,0.199713,0.045968,-1.010827,-0.037244,-1.167648,-0.934562,-1.632222,-1.800113,0.795093,-0.819242,0.341708,0.0,0.0,0.592522,1.030553
2,2024-01-03,1.390394,0.983760,0.858406,-0.363293,0.576359,0.064076,-0.780618,1.353453,-1.126530,0.842011,-0.621855,-1.872609,1.437292,-1.579540,0.251201,0.0,0.0,-0.913281,1.053477
3,2024-01-04,1.839586,1.410013,1.241026,-0.072721,0.454287,-0.424035,-0.079583,0.442307,-1.101687,2.026393,1.443291,-1.813186,1.477354,1.556690,0.009850,0.0,0.0,-1.852031,0.931216
4,2024-01-05,1.108680,0.937322,0.565740,-0.267107,0.062305,0.478107,-0.355315,0.610149,-1.093121,1.611860,0.836330,-1.727617,1.647312,-0.344056,-0.186248,0.0,0.0,-2.264622,0.931216
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,1.140304,0.103580,-0.293255,0.747124,0.342642,-0.142116,-1.377602,-0.348952,-0.677651,-0.579247,-1.302835,-1.318787,1.478568,-1.294428,1.616343,0.0,0.0,-2.547758,-0.948545
362,2024-12-28,0.159792,-0.697347,0.371896,0.302957,0.326693,-0.775508,-1.151294,-0.013266,-1.025018,-0.737165,-0.436806,-1.277191,1.593896,0.036093,-0.751914,0.0,0.0,-2.388869,0.167086
363,2024-12-29,0.124780,-0.609869,-0.957138,-0.893574,-0.980525,-0.702250,-0.999122,-0.612704,-0.417233,-1.210917,-1.610017,-1.323541,1.454288,1.461653,0.990339,0.0,0.0,-1.432146,0.167086
364,2024-12-30,0.148982,-0.560603,-0.900125,-0.809978,-0.913048,-0.991166,-0.990017,-0.324974,-0.146107,-1.270136,-1.543399,-1.485171,1.270369,1.176541,0.681108,0.0,0.0,-1.551704,0.167086


In [10]:
df.to_excel('Ashokvihar2024.xlsx', index=False)